# Member 3: Classification Final Comparison and Model Selection (A2/A3)

This notebook provides Review 2 classification deliverables **A2: one comparison table for all ten Part A + Part B algorithms** and **A3: a justified final model selection with tuning evidence**.

It reuses the same Adult preprocessing and train/test split as the existing notebooks. The model names below match the repository's five Part A models and the two Part B models. The comparison uses the existing saved result tables when available, so metrics are not fabricated or silently recomputed with a different split.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RESULT_DIR = Path("../results/classification")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
print("Working directory should be review2/notebooks/")
print("Results directory:", RESULT_DIR.resolve())

## 1. Collect the existing Part A and Part B evaluation outputs

The Part A notebook's `model_comparison.csv` and Part B notebook's comparison file are the sources of recorded scores. This cell searches likely output filenames and reports any missing inputs. If the files use different names, update the candidate list to the exact repository path rather than entering scores manually.

In [ ]:
candidates = [
    Path("../results/classification/model_comparison.csv"),
    Path("../results/classification/partB_member1_model_comparison.csv"),
    Path("../results/classification/partB_member1_comparison.csv"),
    Path("../results/classification/partB_member1_metrics.csv"),
]
found = [p for p in candidates if p.exists()]
print("Found comparison files:")
for p in found:
    print("-", p)
if not found:
    raise FileNotFoundError("No existing comparison CSV found in ../results/classification/. Run the Part A and Part B notebooks first.")

## 2. Normalize metric column names and combine models

In [ ]:
def load_table(path):
    table = pd.read_csv(path)
    table.columns = [str(c).strip() for c in table.columns]
    return table

tables = [load_table(p).assign(_source=p.name) for p in found]
raw_comparison = pd.concat(tables, ignore_index=True, sort=False)
display(raw_comparison.head())
print("Combined rows:", len(raw_comparison))

## 3. Build the single A2 comparison table

The rubric explicitly asks for Accuracy, Precision, Recall, F1 and ROC-AUC across all ten classifiers. The code maps common column-name variants to a consistent schema and keeps the source columns visible if any required metric cannot be mapped.

In [ ]:
def norm(s):
    return "".join(ch for ch in str(s).lower() if ch.isalnum())

aliases = {
    "model": ["model", "algorithm", "classifier", "modelname"],
    "accuracy": ["accuracy", "testaccuracy"],
    "precision": ["precision", "weightedprecision", "precisionweighted"],
    "recall": ["recall", "weightedrecall", "recallweighted"],
    "f1": ["f1", "weightedf1", "f1weighted", "f1score", "weightedf1score"],
    "roc_auc": ["rocauc", "rocaucscore", "auc", "testrocauc"],
}
col_lookup = {norm(c): c for c in raw_comparison.columns}
rename = {}
for target, opts in aliases.items():
    for opt in opts:
        if opt in col_lookup:
            rename[col_lookup[opt]] = target
            break
comparison = raw_comparison.rename(columns=rename)
required = ["model", "accuracy", "precision", "recall", "f1", "roc_auc"]
missing = [c for c in required if c not in comparison.columns]
if missing:
    print("Could not automatically map required columns:", missing)
    print("Available columns:", list(raw_comparison.columns))
    raise ValueError("Adjust aliases above to match the actual result CSV headers. Do not manually invent missing metrics.")
comparison = comparison[required + (["_source"] if "_source" in comparison.columns else [])].copy()
for c in required[1:]:
    comparison[c] = pd.to_numeric(comparison[c], errors="coerce")
comparison = comparison.drop_duplicates(subset=["model"], keep="last").sort_values("f1", ascending=False)
display(comparison.round(4))
comparison.to_csv(RESULT_DIR / "member3_all_10_classifiers_comparison.csv", index=False)

## 4. A2 completeness check

The team should have five Part A algorithms (Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, SVC) and five Part B algorithms. This cell reports the number of distinct models available. It does not pretend the table is complete when outputs are missing.

In [ ]:
print("Distinct algorithms present:", comparison["model"].nunique())
display(comparison[["model"]].reset_index(drop=True))
if comparison["model"].nunique() < 10:
    print("WARNING: A2 is not yet complete. Run all Part A and Part B notebooks, then update the candidate file list to include every model result.")

## 5. A2 visual comparison

In [ ]:
plot_df = comparison.melt(id_vars="model", value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
                           var_name="Metric", value_name="Score")
plt.figure(figsize=(13, 6))
sns.barplot(data=plot_df, x="model", y="Score", hue="Metric")
plt.xticks(rotation=35, ha="right")
plt.ylim(0, 1)
plt.title("Classification: all available models and metrics")
plt.tight_layout()
plt.savefig(RESULT_DIR / "member3_all_classifiers_metric_comparison.png", dpi=180)
plt.show()

## 6. A3 final model selection and tuning evidence

Select the model using the project's primary metric, weighted F1, while checking ROC-AUC and accuracy. The best row in the combined table is a data-derived candidate, not an automatic claim that it is universally best. The tuning evidence should come from the existing GridSearchCV/CV output files.

In [ ]:
best = comparison.sort_values("f1", ascending=False).iloc[0]
print("Candidate selected by weighted F1:", best["model"])
display(best.to_frame("Selected model result"))
print("Selection basis: highest available weighted F1; use ROC-AUC, recall and the confusion matrix to discuss trade-offs.")

In [ ]:
cv_candidates = [
    Path("../results/classification/cv_results.csv"),
    Path("../results/classification/partB_member1_cv_results.csv"),
    Path("../results/classification/partB_member1_cv_summary.csv"),
]
cv_found = [p for p in cv_candidates if p.exists()]
if cv_found:
    cv_tables = pd.concat([pd.read_csv(p).assign(_source=p.name) for p in cv_found], ignore_index=True, sort=False)
    print("Cross-validation/tuning evidence files:", [p.name for p in cv_found])
    display(cv_tables.head(20))
else:
    print("No CV summary CSV found under expected names. Cite the GridSearchCV best_params_ and before/after metric output from the model-specific notebooks.")

## 7. A3 written conclusion template

After running this notebook, use the actual displayed values to complete the conclusion:
1. Name the model with the highest weighted F1 in the combined table.
2. Quote its Accuracy, Precision, Recall, F1 and ROC-AUC.
3. State which hyperparameters were tuned and the CV scoring metric.
4. Report the before/after metric and the absolute improvement from the model notebook.
5. Explain the trade-off: the Adult dataset is imbalanced, so accuracy alone can hide poor detection of the `>50K` class.

Do not report a model as best until all ten algorithms are present in the table.